# GenQuery Quick Start: SQLite, Sample HR Data, and Charts

This notebook is a beginner-friendly, end-to-end walkthrough for GenQuery.

You will:

1. Install the required packages.
2. Create a local SQLite database.
3. Load the sample HR data from `examples/data.sql`.
4. Ask natural-language questions with GenQuery.
5. Inspect generated SQL.
6. Create charts from the returned data.

SQLite is used so you can run this notebook without starting PostgreSQL, MySQL, MSSQL, or Oracle.

## 1. Install dependencies

Run this cell first. If packages are installed for the first time, restart the notebook kernel before continuing.

In [1]:
%pip install -U genquery pyarrow pandas nbformat>=4.2.0 plotly[express]==6.7.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import packages

In [2]:
from pathlib import Path
import os
import sqlite3

import polars as pl
import plotly.express as px
from sqlalchemy import create_engine, text

## 3. Locate the sample SQL file

The repository already contains sample data at `examples/data.sql`.

In [3]:
PROJECT_ROOT = (Path.cwd()).resolve().absolute()
DB_PATH = PROJECT_ROOT / "quickstart_hr.sqlite"
CONNECTION_STRING = f"sqlite:///{DB_PATH}"

print("Project root:", PROJECT_ROOT)
print("SQLite database:", DB_PATH)
print("Connection string:", CONNECTION_STRING)

Project root: C:\MyProjects\community\gen-query\examples
SQLite database: C:\MyProjects\community\gen-query\examples\quickstart_hr.sqlite
Connection string: sqlite:///C:\MyProjects\community\gen-query\examples\quickstart_hr.sqlite


## 4. Create a local SQLite database

In [4]:
sqlite_data_sql = (PROJECT_ROOT  / "data.sql").read_text()
with sqlite3.connect(DB_PATH) as conn:
    conn.executescript(sqlite_data_sql)

print("Sample HR data loaded into SQLite.")

Sample HR data loaded into SQLite.


## 5. Validate the database

Before using GenQuery, confirm that each table contains rows.

In [5]:
engine = create_engine(CONNECTION_STRING)

tables = [
    "regions",
    "countries",
    "locations",
    "jobs",
    "departments",
    "employees",
    "dependents",
]

with engine.connect() as conn:
    for table in tables:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"{table:12s}: {count}")

regions     : 4
countries   : 25
locations   : 7
jobs        : 19
departments : 11
employees   : 40
dependents  : 30


## 6. Preview the data with normal SQL

This step is optional, but it helps you understand the sample HR database.

In [6]:
preview_sql = """
SELECT
    e.employee_id,
    e.first_name || ' ' || e.last_name AS employee_name,
    d.department_name,
    j.job_title,
    e.salary
FROM employees e
LEFT JOIN departments d ON e.department_id = d.department_id
LEFT JOIN jobs j ON e.job_id = j.job_id
ORDER BY e.salary DESC
LIMIT 10
"""

with engine.connect() as conn:
    rows = conn.execute(text(preview_sql)).fetchall()

for row in rows:
    print(row)

(100, 'Steven King', 'Executive', 'President', 24000)
(101, 'Neena Kochhar', 'Executive', 'Administration Vice President', 17000)
(102, 'Lex De Haan', 'Executive', 'Administration Vice President', 17000)
(145, 'John Russell', 'Sales', 'Sales Manager', 14000)
(146, 'Karen Partners', 'Sales', 'Sales Manager', 13500)
(201, 'Michael Hartstein', 'Marketing', 'Marketing Manager', 13000)
(108, 'Nancy Greenberg', 'Finance', 'Finance Manager', 12000)
(205, 'Shelley Higgins', 'Accounting', 'Accounting Manager', 12000)
(114, 'Den Raphaely', 'Purchasing', 'Purchasing Manager', 11000)
(204, 'Hermann Baer', 'Public Relations', 'Public Relations Representative', 10000)


## 7. Configure your LLM adapter

GenQuery needs an LLM to translate natural language into SQL. The built-in `OpenAIAdapter` works with OpenAI and OpenAI-compatible APIs.

### Option A: OpenAI

Set these environment variables before running the notebook:

```bash
OPENAI_API_KEY=your_openai_key
GENQUERY_MODEL=gpt-4o-mini
```

### Option B: OpenRouter or another OpenAI-compatible endpoint

```bash
OPENROUTER_API_KEY=your_openrouter_key
GENQUERY_BASE_URL=https://openrouter.ai/api/v1
GENQUERY_MODEL=openai/gpt-oss-120b:free
```

In [7]:
from genquery import GenQuery
from genquery.adapters.openai_adapter import OpenAIAdapter

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY")

if not api_key:
    raise RuntimeError(
        "Set OPENAI_API_KEY or OPENROUTER_API_KEY before running GenQuery cells."
    )

llm = OpenAIAdapter(
    api_key=api_key,
    model=os.getenv("GENQUERY_MODEL", "openai/gpt-oss-120b:free"),
    base_url=os.getenv("GENQUERY_BASE_URL", "https://openrouter.ai/api/v1"),
)

gq = GenQuery(
    llm=llm,
    connection_string=CONNECTION_STRING,
    schema="main",
    log_level="INFO",
)

gq

2026-05-31 16:55:19,829 INFO [genquery.genquery] Initializing GenQuery
2026-05-31 16:55:19,832 INFO [genquery.genquery] GenQuery initialized for dialect=sqlite


GenQuery(dialect=sqlite, schema=main, pipeline_stages=4, llm=OpenAIAdapter)

## 8. Ask your first natural-language question

Use `return_result=True` when you want the generated SQL, query plan, returned DataFrame, and conversation history.

In [8]:
result = gq.run(
    "Show the top 5 employees by salary. Include employee_name, department, job_title, and salary.",
    return_result=True,
)

print("Generated SQL:")
print(result.sql)

result.df

Generated SQL:
SELECT e.first_name || ' ' || e.last_name AS full_name,        d.department_name,        j.job_title,        e.salary FROM employees e JOIN departments d ON e.department_id = d.department_id JOIN jobs j ON e.job_id = j.job_id ORDER BY e.salary DESC LIMIT 5;


full_name,department_name,job_title,salary
str,str,str,i64
"""Steven King""","""Executive""","""President""",24000
"""Neena Kochhar""","""Executive""","""Administration Vice President""",17000
"""Lex De Haan""","""Executive""","""Administration Vice President""",17000
"""John Russell""","""Sales""","""Sales Manager""",14000
"""Karen Partners""","""Sales""","""Sales Manager""",13500


## 9. Total salary by department

For charting, it is useful to ask GenQuery to return specific column names.

In [9]:
result = gq.run(
    "Show total salary by department. Return columns named department and total_salary.",
    return_result=True,
)

print("Generated SQL:")
print(result.sql)

df_salary_by_department = result.df.sort("total_salary", descending=True)
df_salary_by_department

Generated SQL:
SELECT d.department_name AS department, SUM(e.salary) AS total_salary FROM employees e JOIN departments d ON e.department_id = d.department_id GROUP BY d.department_name;


department,total_salary
str,i64
"""Executive""",58000
"""Sales""",57700
"""Finance""",51600
"""Shipping""",41200
"""IT""",28800
…,…
"""Accounting""",20300
"""Marketing""",19000
"""Public Relations""",10000


## 10. Create a Plotly bar chart

In [10]:
fig = px.bar(
    df_salary_by_department.to_pandas(),
    x="department",
    y="total_salary",
    title="Total Salary by Department",
    labels={"department": "Department", "total_salary": "Total Salary"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 11. Average salary by job title

In [11]:
result = gq.run(
    "Show average salary by job title. Return columns named job_title and average_salary.",
    return_result=True,
)

print("Generated SQL:")
print(result.sql)

df_avg_salary_by_job = result.df.sort("average_salary", descending=True)
df_avg_salary_by_job

Generated SQL:
SELECT j.job_title, AVG(e.salary) AS average_salary FROM employees e JOIN jobs j ON e.job_id = j.job_id GROUP BY j.job_title;


job_title,average_salary
str,f64
"""President""",24000.0
"""Administration Vice President""",17000.0
"""Sales Manager""",13750.0
"""Marketing Manager""",13000.0
"""Accounting Manager""",12000.0
…,…
"""Programmer""",5760.0
"""Administration Assistant""",4400.0
"""Shipping Clerk""",3950.0


In [12]:
fig = px.bar(
    df_avg_salary_by_job.to_pandas(),
    x="job_title",
    y="average_salary",
    title="Average Salary by Job Title",
    labels={"job_title": "Job Title", "average_salary": "Average Salary"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 12. Employee count by department

In [13]:
result = gq.run(
    "Count employees by department. Return columns named department and employee_count.",
    return_result=True,
)

print("Generated SQL:")
print(result.sql)

df_employee_count = result.df.sort("employee_count", descending=True)
df_employee_count

Generated SQL:
SELECT d.department_name AS department,        COUNT(e.employee_id) AS employee_count FROM employees e JOIN departments d ON e.department_id = d.department_id GROUP BY d.department_name;


department,employee_count
str,i64
"""Shipping""",7
"""Finance""",6
"""Purchasing""",6
"""Sales""",6
"""IT""",5
…,…
"""Accounting""",2
"""Marketing""",2
"""Administration""",1


In [14]:
fig = px.pie(
    df_employee_count.to_pandas(),
    names="department",
    values="employee_count",
    title="Employee Count by Department",
)
fig.show()

## 13. Dry run: inspect generated SQL safely

`dry_run()` generates SQL and validates it with `EXPLAIN`, without running the final SQL as a normal data-returning query. This is useful when you want to review generated SQL before using it.

In [15]:
dry_result = gq.dry_run(
    "Which departments have the highest average salary?"
)

print("Generated SQL:")
print(dry_result.sql)

print("\nPlan:")
print(dry_result.plan)

Generated SQL:
SELECT d.* FROM departments d JOIN (     SELECT department_id, AVG(salary) AS avg_salary     FROM employees     GROUP BY department_id ) AS dept_avg ON d.department_id = dept_avg.department_id WHERE dept_avg.avg_salary = (     SELECT MAX(avg_salary)     FROM (         SELECT AVG(salary) AS avg_salary         FROM employees         GROUP BY department_id     ) );

Plan:
strategy='sequential' steps=[PlanStep(id='step_1', description='Calculate the average salary for each department.', depends_on=[], output_alias='dept_avg_salary', receives_context=None), PlanStep(id='step_2', description='Find the maximum average salary value from the previous result.', depends_on=['step_1'], output_alias='max_avg_salary', receives_context='dept_avg_salary'), PlanStep(id='step_3', description="Select department details where the department's average salary equals the maximum average salary.", depends_on=['step_1', 'step_2'], output_alias='result', receives_context='dept_avg_salary, max_avg

## 14. Multi-turn conversation

Pass the previous result's `conversation` into the next call so GenQuery can understand follow-up questions.

In [16]:
first = gq.generate(
    "Show average salary by department. Return columns named department and average_salary."
)

print("Generated SQL:")
print(first.sql)

first.df

Generated SQL:
SELECT d.department_name AS department,        AVG(e.salary) AS average_salary FROM employees e JOIN departments d ON e.department_id = d.department_id GROUP BY d.department_name;


department,average_salary
str,f64
"""Accounting""",10150.0
"""Administration""",4400.0
"""Executive""",19333.333333
"""Finance""",8600.0
"""Human Resources""",6500.0
…,…
"""Marketing""",9500.0
"""Public Relations""",10000.0
"""Purchasing""",4150.0


In [17]:
follow_up = gq.generate(
    "Now show only departments where the average salary is greater than 8000.",
    conversation=first.conversation,
)

print("Generated SQL:")
print(follow_up.sql)

follow_up.df

Generated SQL:
SELECT d.department_name AS department,        AVG(e.salary) AS average_salary FROM employees e JOIN departments d ON e.department_id = d.department_id GROUP BY d.department_name HAVING AVG(e.salary) > 8000;


department,average_salary
str,f64
"""Accounting""",10150.0
"""Executive""",19333.333333
"""Finance""",8600.0
"""Marketing""",9500.0
"""Public Relations""",10000.0
"""Sales""",9616.666667


## 15. Streaming results

Use `stream()` when the final result may be large. The stream yields Polars DataFrame batches.

In [18]:
stream_result = gq.stream(
    "Show all employees with their department and job title.",
    batch_size=10,
)

print("Generated SQL:")
print(stream_result.sql)

with stream_result.stream as batches:
    for batch in batches:
        display(batch)

Generated SQL:
SELECT e.employee_id,        e.first_name,        e.last_name,        e.email,        e.phone_number,        e.hire_date,        e.job_id,        e.salary,        e.manager_id,        e.department_id,        d.department_name,        j.job_title FROM employees e JOIN departments d ON e.department_id = d.department_id JOIN jobs j ON e.job_id = j.job_id;


employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,manager_id,department_id,department_name,job_title
i64,str,str,str,str,str,i64,i64,i64,i64,str,str
100,"""Steven""","""King""","""steven.king@sqltutorial.org""","""515.123.4567""","""2015-06-17""",4,24000,null,9,"""Executive""","""President"""
101,"""Neena""","""Kochhar""","""neena.kochhar@sqltutorial.org""","""515.123.4568""","""2016-09-21""",5,17000,100,9,"""Executive""","""Administration Vice President"""
102,"""Lex""","""De Haan""","""lex.de haan@sqltutorial.org""","""515.123.4569""","""2019-01-13""",5,17000,100,9,"""Executive""","""Administration Vice President"""
103,"""Alexander""","""Hunold""","""alexander.hunold@sqltutorial.o…","""590.423.4567""","""2018-01-03""",9,9000,102,6,"""IT""","""Programmer"""
104,"""Bruce""","""Ernst""","""bruce.ernst@sqltutorial.org""","""590.423.4568""","""2019-05-21""",9,6000,103,6,"""IT""","""Programmer"""
105,"""David""","""Austin""","""david.austin@sqltutorial.org""","""590.423.4569""","""2023-06-25""",9,4800,103,6,"""IT""","""Programmer"""
106,"""Valli""","""Pataballa""","""valli.pataballa@sqltutorial.or…","""590.423.4560""","""2024-02-05""",9,4800,103,6,"""IT""","""Programmer"""
107,"""Diana""","""Lorentz""","""diana.lorentz@sqltutorial.org""","""590.423.5567""","""2025-02-07""",9,4200,103,6,"""IT""","""Programmer"""
108,"""Nancy""","""Greenberg""","""nancy.greenberg@sqltutorial.or…","""515.124.4569""","""2020-08-17""",7,12000,101,10,"""Finance""","""Finance Manager"""


employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,manager_id,department_id,department_name,job_title
i64,str,str,str,str,str,i64,i64,i64,i64,str,str
110,"""John""","""Chen""","""john.chen@sqltutorial.org""","""515.124.4269""","""2023-09-28""",6,8200,108,10,"""Finance""","""Accountant"""
111,"""Ismael""","""Sciarra""","""ismael.sciarra@sqltutorial.org""","""515.124.4369""","""2023-09-30""",6,7700,108,10,"""Finance""","""Accountant"""
112,"""Jose Manuel""","""Urman""","""jose manuel.urman@sqltutorial.…","""515.124.4469""","""2024-03-07""",6,7800,108,10,"""Finance""","""Accountant"""
113,"""Luis""","""Popp""","""luis.popp@sqltutorial.org""","""515.124.4567""","""2025-12-07""",6,6900,108,10,"""Finance""","""Accountant"""
114,"""Den""","""Raphaely""","""den.raphaely@sqltutorial.org""","""515.127.4561""","""2020-12-07""",14,11000,100,3,"""Purchasing""","""Purchasing Manager"""
115,"""Alexander""","""Khoo""","""alexander.khoo@sqltutorial.org""","""515.127.4562""","""2021-05-18""",13,3100,114,3,"""Purchasing""","""Purchasing Clerk"""
116,"""Shelli""","""Baida""","""shelli.baida@sqltutorial.org""","""515.127.4563""","""2023-12-24""",13,2900,114,3,"""Purchasing""","""Purchasing Clerk"""
117,"""Sigal""","""Tobias""","""sigal.tobias@sqltutorial.org""","""515.127.4564""","""2023-07-24""",13,2800,114,3,"""Purchasing""","""Purchasing Clerk"""
118,"""Guy""","""Himuro""","""guy.himuro@sqltutorial.org""","""515.127.4565""","""2024-11-15""",13,2600,114,3,"""Purchasing""","""Purchasing Clerk"""


employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,manager_id,department_id,department_name,job_title
i64,str,str,str,str,str,i64,i64,i64,i64,str,str
120,"""Matthew""","""Weiss""","""matthew.weiss@sqltutorial.org""","""650.123.1234""","""2022-07-18""",19,8000,100,5,"""Shipping""","""Stock Manager"""
121,"""Adam""","""Fripp""","""adam.fripp@sqltutorial.org""","""650.123.2234""","""2023-04-10""",19,8200,100,5,"""Shipping""","""Stock Manager"""
122,"""Payam""","""Kaufling""","""payam.kaufling@sqltutorial.org""","""650.123.3234""","""2021-05-01""",19,7900,100,5,"""Shipping""","""Stock Manager"""
123,"""Shanta""","""Vollman""","""shanta.vollman@sqltutorial.org""","""650.123.4234""","""2023-10-10""",19,6500,100,5,"""Shipping""","""Stock Manager"""
126,"""Irene""","""Mikkilineni""","""irene.mikkilineni@sqltutorial.…","""650.124.1224""","""2024-09-28""",18,2700,120,5,"""Shipping""","""Stock Clerk"""
145,"""John""","""Russell""","""john.russell@sqltutorial.org""",null,"""2022-10-01""",15,14000,100,8,"""Sales""","""Sales Manager"""
146,"""Karen""","""Partners""","""karen.partners@sqltutorial.org""",null,"""2023-01-05""",15,13500,100,8,"""Sales""","""Sales Manager"""
176,"""Jonathon""","""Taylor""","""jonathon.taylor@sqltutorial.or…",null,"""2024-03-24""",16,8600,100,8,"""Sales""","""Sales Representative"""
177,"""Jack""","""Livingston""","""jack.livingston@sqltutorial.or…",null,"""2024-04-23""",16,8400,100,8,"""Sales""","""Sales Representative"""


employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,manager_id,department_id,department_name,job_title
i64,str,str,str,str,str,i64,i64,i64,i64,str,str
179,"""Charles""","""Johnson""","""charles.johnson@sqltutorial.or…",null,"""2026-01-04""",16,6200,100,8,"""Sales""","""Sales Representative"""
192,"""Sarah""","""Bell""","""sarah.bell@sqltutorial.org""","""650.501.1876""","""2022-02-04""",17,4000,123,5,"""Shipping""","""Shipping Clerk"""
193,"""Britney""","""Everett""","""britney.everett@sqltutorial.or…","""650.501.2876""","""2023-03-03""",17,3900,123,5,"""Shipping""","""Shipping Clerk"""
200,"""Jennifer""","""Whalen""","""jennifer.whalen@sqltutorial.or…","""515.123.4444""","""2015-09-17""",3,4400,101,1,"""Administration""","""Administration Assistant"""
201,"""Michael""","""Hartstein""","""michael.hartstein@sqltutorial.…","""515.123.5555""","""2022-02-17""",10,13000,100,2,"""Marketing""","""Marketing Manager"""
202,"""Pat""","""Fay""","""pat.fay@sqltutorial.org""","""603.123.6666""","""2023-08-17""",11,6000,201,2,"""Marketing""","""Marketing Representative"""
203,"""Susan""","""Mavris""","""susan.mavris@sqltutorial.org""","""515.123.7777""","""2020-06-07""",8,6500,101,4,"""Human Resources""","""Human Resources Representative"""
204,"""Hermann""","""Baer""","""hermann.baer@sqltutorial.org""","""515.123.8888""","""2020-06-07""",12,10000,101,7,"""Public Relations""","""Public Relations Representativ…"
205,"""Shelley""","""Higgins""","""shelley.higgins@sqltutorial.or…","""515.123.8080""","""2020-06-07""",2,12000,101,11,"""Accounting""","""Accounting Manager"""


## Done

You have completed the GenQuery quick start. You created a local SQLite database, loaded sample HR data, asked natural-language questions, inspected generated SQL, and built charts from GenQuery results.

Next ideas:

- Try your own questions against the HR data.
- Replace SQLite with PostgreSQL, MySQL, MSSQL, or Oracle.
- Use `dry_run()` before executing important queries.
- Add table filters or row-level-security policies through `GenQueryConfig`.